# Probability Discounting: Joint Choice and Reaction-Time Models

This notebook is the **single experiment console** for the project.

Core model, likelihood, fitting, recovery, reliability, evaluation, and provenance logic remains in
`src/pd_project/` and `scripts/`. The notebook orchestrates those components, displays diagnostics,
and presents final outputs without duplicating the scientific implementation.

> **Formal Run-B rule:** Section 9 must not be executed until Gate 2 is fully frozen.
> Once the one-shot Run-B registry is `completed`, the notebook must reuse those artifacts rather than rerun the formal evaluation.

## 0. Experiment Control

In [1]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config" / "analysis.yaml").exists():
    raise RuntimeError(
        "Open final_project.ipynb from the repository root in VS Code."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pd_project.config import load_config

CONFIG_PATH = PROJECT_ROOT / "config" / "analysis.yaml"
config = load_config(CONFIG_PATH)

print("Project root:", PROJECT_ROOT)
print("Freeze status:", config["project"]["freeze_status"])
print("Formal Run B enabled:", config["project"]["formal_run_b_enabled"])
print("Master seed:", config["random"]["master_seed"])
print("Enabled models:",
      [m for m in ("M1", "M2", "M3") if config["models"][m]["enabled"]])
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Project root: /Users/zhengbinheng/Desktop/probability-discounting-choice-rt
Freeze status: frozen
Formal Run B enabled: True
Master seed: 2026001
Enabled models: ['M1', 'M2', 'M3']
Python: 3.11.15
Platform: macOS-26.5.2-arm64-arm-64bit
NumPy: 1.26.4
Pandas: 2.2.3


In [2]:
# Explicit execution switches.
# Keep expensive/formal stages False unless you intentionally want to run them.

RUN_PREPARE_DATA = False
RUN_TESTS = False
RUN_SMOKE_RECOVERY = False
RUN_RUN_A_FIT = False
RUN_FORMAL_RECOVERY = False

# Formal Run-B should normally remain False.
# Turn it on only after completing Gate 2 and reviewing the preflight section.
RUN_FORMAL_RUN_B = True

RUN_DERIVED_OUTPUTS = False

EXPERIMENT_MODE = {
    "prepare_data": RUN_PREPARE_DATA,
    "tests": RUN_TESTS,
    "smoke_recovery": RUN_SMOKE_RECOVERY,
    "run_a_fit": RUN_RUN_A_FIT,
    "formal_recovery": RUN_FORMAL_RECOVERY,
    "formal_run_b": RUN_FORMAL_RUN_B,
    "derived_outputs": RUN_DERIVED_OUTPUTS,
}
EXPERIMENT_MODE

{'prepare_data': False,
 'tests': False,
 'smoke_recovery': False,
 'run_a_fit': False,
 'formal_recovery': False,
 'formal_run_b': True,
 'derived_outputs': False}

## 1. Data Preparation

In [6]:
def run_command(args):
    print("$", " ".join(map(str, args)))
    completed = subprocess.run(
        list(map(str, args)),
        cwd=PROJECT_ROOT,
        check=True,
        text=True,
    )
    return completed.returncode

if RUN_PREPARE_DATA:
    run_command([
        sys.executable,
        "scripts/prepare_data.py",
        "--config",
        "config/analysis.yaml",
    ])
else:
    print("Data preparation skipped. Set RUN_PREPARE_DATA=True to execute.")

Data preparation skipped. Set RUN_PREPARE_DATA=True to execute.


In [19]:
processed_path = PROJECT_ROOT / config["data"]["processed_trials"]
audit_path = PROJECT_ROOT / config["data"]["audit_report"]

if not processed_path.exists() or not audit_path.exists():
    raise FileNotFoundError(
        "Prepared data are missing. Run Section 1 after placing PD data.zip in data/raw/."
    )

trials = pd.read_csv(processed_path, dtype={"participant": "string"})
audit = json.loads(audit_path.read_text(encoding="utf-8"))

print(f"Prepared trials: {len(trials):,}")
print("Participants:", trials["participant"].nunique())
print("Audit approved for fitting:", audit.get("approved_for_fitting"))
print("Passed integrity checks:", audit.get("passed_integrity_checks"))
print("Passed frozen contract:", audit.get("passed_frozen_contract"))
print("Processed SHA256:", audit.get("processed_data_sha256"))

Prepared trials: 19,600
Participants: 49
Audit approved for fitting: True
Passed integrity checks: True
Passed frozen contract: True
Processed SHA256: e2bb506ef9bd17c4b8e062646715087a07621486b7af1360eb8ceed5703b1835


In [20]:
# Basic descriptives
basic = pd.DataFrame({
    "quantity": [
        "participants",
        "total_trials",
        "valid_choice_trials",
        "valid_rt_trials",
        "missing_choice_trials",
    ],
    "value": [
        trials["participant"].nunique(),
        len(trials),
        int(trials["choice_included"].sum()),
        int(trials["rt_included"].sum()),
        int((~trials["choice_included"]).sum()),
    ],
})
basic

,quantity,value
0,participants,49
1,total_trials,19600
2,valid_choice_trials,19532
3,valid_rt_trials,19532
4,missing_choice_trials,68


## 2. Data Exploration

In [21]:
# Trial counts by run and condition
trial_counts = (
    trials.groupby(["run", "condition"], as_index=False)
    .size()
    .rename(columns={"size": "n_trials"})
)
trial_counts

,run,condition,n_trials
0,A,L,4900
1,A,R,4900
2,B,L,4900
3,B,R,4900


In [22]:
# Choice proportions by run and condition
choice_desc = (
    trials.loc[trials["choice_included"]]
    .groupby(["run", "condition"], as_index=False)
    .agg(
        n=("choice_uncertain", "size"),
        p_uncertain=("choice_uncertain", "mean"),
    )
)
choice_desc

,run,condition,n,p_uncertain
0,A,L,4860,0.385597
1,A,R,4889,0.402127
2,B,L,4889,0.479853
3,B,R,4894,0.525746


In [23]:
# RT descriptives by run and condition
rt_desc = (
    trials.loc[trials["rt_included"]]
    .groupby(["run", "condition"])["rt_seconds"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)
rt_desc

,run,condition,count,mean,median,std,min,max
0,A,L,4860,2.730717,2.39075,1.524193,0.2067,9.9292
1,A,R,4889,2.298525,2.00670,1.312518,0.2000,9.9148
2,B,L,4889,2.059968,1.79550,1.277005,0.2013,9.7872
3,B,R,4894,1.853597,1.64170,1.151083,0.2000,9.9963


In [24]:
# Predictor distributions under a neutral k=1 reference.
# This is descriptive only; fitted predictors are calculated later from participant estimates.

from pd_project.valuation_choice import subjective_values
from pd_project.rt_models import rt_predictor

s0 = float(config["data"]["transforms"]["amount_scale"])
reference = trials.loc[trials["run"] == "A"].copy()
values = subjective_values(
    reference["r_cert"].to_numpy(float),
    reference["r_uncert"].to_numpy(float),
    reference["odds"].to_numpy(float),
    np.ones(len(reference)),
    s0=s0,
)

predictor_rows = []
for model in ("M1", "M2", "M3"):
    g = rt_predictor(model, values.v_cert, values.v_uncert, values.delta_v)
    temp = pd.DataFrame({
        "model": model,
        "condition": reference["condition"].to_numpy(),
        "predictor": g,
    })
    predictor_rows.append(temp)

predictors = pd.concat(predictor_rows, ignore_index=True)
predictor_summary = (
    predictors.groupby(["model", "condition"])["predictor"]
    .quantile([0.00, 0.50, 0.90, 0.95, 0.99, 1.00])
    .rename("value")
    .reset_index()
    .rename(columns={"level_2": "quantile"})
)
predictor_summary

,model,condition,quantile,value
0,M1,L,0.00,0.000000
1,M1,L,0.50,0.449675
2,M1,L,0.90,6.825000
3,M1,L,0.95,11.362738
4,M1,L,0.99,33.817500
5,M1,L,1.00,40.500000
6,M1,R,0.00,0.000000
7,M1,R,0.50,0.449675
8,M1,R,0.90,6.825000
9,M1,R,0.95,11.362738


## 3. Synthetic Smoke Test

In [25]:
if RUN_TESTS:
    run_command([sys.executable, "-m", "pytest", "-q"])
else:
    print("pytest skipped. Set RUN_TESTS=True to execute.")

pytest skipped. Set RUN_TESTS=True to execute.


In [26]:
if RUN_SMOKE_RECOVERY:
    run_command([
        sys.executable,
        "scripts/run_recovery.py",
        "--config",
        "config/analysis.yaml",
        "--smoke",
    ])
else:
    print("Synthetic smoke recovery skipped. Set RUN_SMOKE_RECOVERY=True to execute.")

Synthetic smoke recovery skipped. Set RUN_SMOKE_RECOVERY=True to execute.


## 4. Run-A Model Fitting

In [ ]:
if RUN_RUN_A_FIT:
    run_command([
        sys.executable,
        "scripts/fit_run_a.py",
        "--config",
        "config/analysis.yaml",
    ])
else:
    print("Run-A fitting skipped. Set RUN_RUN_A_FIT=True to execute.")

In [ ]:
run_a_fits_path = PROJECT_ROOT / "results" / "run_a_fits.csv"
run_a_starts_path = PROJECT_ROOT / "results" / "run_a_optimizer_starts.csv"
run_a_receipt_path = PROJECT_ROOT / "results" / "run_a_completion.json"

if run_a_fits_path.exists():
    run_a_fits = pd.read_csv(run_a_fits_path, dtype={"participant": "string"})
    display(run_a_fits.head())
    print("Run-A fit rows:", len(run_a_fits))
else:
    run_a_fits = None
    print("Run-A fits not available yet.")

if run_a_starts_path.exists():
    run_a_starts = pd.read_csv(run_a_starts_path, dtype={"participant": "string"})
    print("Optimizer start rows:", len(run_a_starts))
else:
    run_a_starts = None

,participant,model,choice_only,objective,success,best_start_index,message,log_k_R,log_k_L,log_beta,...,delta_L,log_b,log_sigma,fit_run,multistarts_used,config_sha256,processed_data_sha256,data_pipeline_sha256,git_commit,runtime_sha256
0,2rrjzhgj9,choice_only,True,111.029468,True,7,CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH,0.413516,0.059811,-0.366148,...,NaN,NaN,NaN,A,20,2b171995d7ba6736f44f922371a8192b0631c5e513bcdf...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,ad8b23c6b44ff33f2658dca31d7f9a47e68d03d6357923...,09f1fd60ccd129bef0b43db305f6c2803e2eb490,1daf8f87b03b0ee44e5a7bd43036083ee3ecc0b2cb65e2...
1,3lmzwis6r,choice_only,True,101.839149,True,2,CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH,1.154441,0.035314,0.236874,...,NaN,NaN,NaN,A,20,2b171995d7ba6736f44f922371a8192b0631c5e513bcdf...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,ad8b23c6b44ff33f2658dca31d7f9a47e68d03d6357923...,09f1fd60ccd129bef0b43db305f6c2803e2eb490,1daf8f87b03b0ee44e5a7bd43036083ee3ecc0b2cb65e2...
2,4f8msviwg,choice_only,True,101.725649,True,1,CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH,-0.463484,-4.605170,-0.736657,...,NaN,NaN,NaN,A,20,2b171995d7ba6736f44f922371a8192b0631c5e513bcdf...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,ad8b23c6b44ff33f2658dca31d7f9a47e68d03d6357923...,09f1fd60ccd129bef0b43db305f6c2803e2eb490,1daf8f87b03b0ee44e5a7bd43036083ee3ecc0b2cb65e2...
3,52b64k0rp,choice_only,True,134.724367,True,11,CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH,4.605170,0.983620,-1.732131,...,NaN,NaN,NaN,A,20,2b171995d7ba6736f44f922371a8192b0631c5e513bcdf...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,ad8b23c6b44ff33f2658dca31d7f9a47e68d03d6357923...,09f1fd60ccd129bef0b43db305f6c2803e2eb490,1daf8f87b03b0ee44e5a7bd43036083ee3ecc0b2cb65e2...
4,5wlmihb4c,choice_only,True,115.808091,True,13,CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH,2.108233,-0.338893,-0.460054,...,NaN,NaN,NaN,A,20,2b171995d7ba6736f44f922371a8192b0631c5e513bcdf...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,ad8b23c6b44ff33f2658dca31d7f9a47e68d03d6357923...,09f1fd60ccd129bef0b43db305f6c2803e2eb490,1daf8f87b03b0ee44e5a7bd43036083ee3ecc0b2cb65e2...


Run-A fit rows: 196
Optimizer start rows: 3920


## 5. Run-A Diagnostics

In [ ]:
# 5.1 Convergence diagnostics

if run_a_fits is None:
    raise RuntimeError("Run-A fits are not available. Run Section 4 first.")

success_summary = (
    run_a_fits.groupby("model", as_index=False)
    .agg(
        n_fits=("success", "size"),
        n_success=("success", "sum"),
    )
)

success_summary["failure_rate"] = (
    1.0 - success_summary["n_success"] / success_summary["n_fits"]
)

display(success_summary)

overall_nonconvergence_rate = float(
    (~run_a_fits["success"].astype(bool)).mean()
)

threshold = float(
    config["map_fallback"]["global_triggers"]["nonconvergence_rate_gt"]
)

print("Overall non-convergence rate:", overall_nonconvergence_rate)
print("Trigger threshold:", threshold)
print("Trigger fired:", overall_nonconvergence_rate > threshold)

,model,n_fits,n_success,failure_rate
0,M1,49,49,0.0
1,M2,49,49,0.0
2,M3,49,49,0.0
3,choice_only,49,49,0.0


Overall non-convergence rate: 0.0
Trigger threshold: 0.05
Trigger fired: False


In [ ]:
# 5.2 Parameter distributions

parameter_columns = [
    c for c in [
        "log_k_R",
        "log_k_L",
        "log_beta",
        "alpha",
        "delta_L",
        "log_b",
        "log_sigma",
    ]
    if c in run_a_fits.columns
]

parameter_summary = (
    run_a_fits.loc[run_a_fits["success"].astype(bool)]
    .groupby("model")[parameter_columns]
    .agg(["mean", "std", "min", "median", "max"])
)

display(parameter_summary)

log_k_R                                          log_k_L  \
                 mean       std       min    median      max      mean   
model                                                                    
M1           1.493951  1.988356 -4.605170  1.376863  4.60517 -0.521260   
M2           1.570549  1.831173 -2.798483  1.439302  4.60517 -0.217219   
M3           1.428035  1.941625 -4.605170  1.378929  4.60517 -0.489107   
choice_only  1.526278  1.853630 -4.605170  1.522719  4.60517 -0.465623   

                                                     ...     log_b            \
                  std       min    median       max  ...      mean       std   
model                                                ...                       
M1           1.796927 -4.605170 -0.028068  2.302585  ... -6.501180  2.048849   
M2           1.225967 -2.999884 -0.019717  1.814287  ... -9.878040  1.945724   
M3           1.922795 -4.605170 -0.030796  4.310446  ... -6.564128  2.076907   
choice_only  1.712792 -4.605170 -0.019970  1.826262  ...       NaN       NaN   

                                            log_sigma                      \
                   min     median       max      mean       std       min   
model                                                                       
M1           -9.210340  -6.071311 -2.953389 -0.886247  0.262226 -1.335427   
M2          -11.512925 -10.654141 -4.484141 -0.883146  0.261132 -1.331226   
M3           -9.210340  -5.675347 -3.397175 -0.884040  0.261988 -1.336005   
choice_only        NaN        NaN       NaN       NaN       NaN       NaN   

                                 
               median       max  
model                            
M1          -0.890853  0.293519  
M2          -0.881620  0.293559  
M3          -0.890841  0.293412  
choice_only       NaN       NaN  

[4 rows x 35 columns]

In [ ]:
# 5.3 Parameter-specific boundary diagnostics

bounds = config["optimization"]["bounds"]

boundary_rows = []

for _, row in run_a_fits.iterrows():
    model = str(row["model"])

    params = ["log_k_R", "log_k_L", "log_beta"]

    if model != "choice_only":
        params += ["alpha", "delta_L", "log_b", "log_sigma"]

    for parameter in params:

        if parameter == "log_b":
            lower, upper = bounds["log_b"][model]
        else:
            lower, upper = bounds[parameter]

        estimate = float(row[parameter])

        # "Near boundary" = within 1% of the allowed interval
        tolerance = 0.01 * (upper - lower)

        near_lower = estimate <= lower + tolerance
        near_upper = estimate >= upper - tolerance

        boundary_rows.append(
            {
                "model": model,
                "participant": row["participant"],
                "parameter": parameter,
                "estimate": estimate,
                "lower_bound": lower,
                "upper_bound": upper,
                "near_lower": near_lower,
                "near_upper": near_upper,
                "near_boundary": near_lower or near_upper,
            }
        )

boundary_detail = pd.DataFrame(boundary_rows)

boundary_rates = (
    boundary_detail
    .groupby(["model", "parameter"], as_index=False)
    .agg(
        n=("participant", "size"),
        n_boundary=("near_boundary", "sum"),
        boundary_rate=("near_boundary", "mean"),
        n_lower=("near_lower", "sum"),
        n_upper=("near_upper", "sum"),
    )
)

display(boundary_rates)

,model,parameter,n,n_boundary,boundary_rate,n_lower,n_upper
0,M1,alpha,49,0,0.000000,0,0
1,M1,delta_L,49,0,0.000000,0,0
2,M1,log_b,49,15,0.306122,15,0
3,M1,log_beta,49,2,0.040816,1,1
4,M1,log_k_L,49,5,0.102041,5,0
5,M1,log_k_R,49,10,0.204082,1,9
6,M1,log_sigma,49,0,0.000000,0,0
7,M2,alpha,49,0,0.000000,0,0
8,M2,delta_L,49,0,0.000000,0,0
9,M2,log_b,49,20,0.408163,20,0


In [ ]:
# 5.4 Shared-parameter boundary trigger check

shared_parameters = ["log_k_R", "log_k_L", "log_beta"]

shared_boundary = boundary_rates[
    boundary_rates["parameter"].isin(shared_parameters)
].copy()

shared_boundary["trigger_threshold"] = float(
    config["map_fallback"]["global_triggers"]["shared_parameter_boundary_rate_gt"]
)

shared_boundary["trigger_fired"] = (
    shared_boundary["boundary_rate"]
    > shared_boundary["trigger_threshold"]
)

display(shared_boundary)

,model,parameter,n,n_boundary,boundary_rate,n_lower,n_upper,trigger_threshold,trigger_fired
3,M1,log_beta,49,2,0.040816,1,1,0.05,False
4,M1,log_k_L,49,5,0.102041,5,0,0.05,True
5,M1,log_k_R,49,10,0.204082,1,9,0.05,True
10,M2,log_beta,49,2,0.040816,1,1,0.05,False
11,M2,log_k_L,49,0,0.000000,0,0,0.05,False
12,M2,log_k_R,49,9,0.183673,0,9,0.05,True
17,M3,log_beta,49,2,0.040816,1,1,0.05,False
18,M3,log_k_L,49,6,0.122449,6,0,0.05,True
19,M3,log_k_R,49,8,0.163265,1,7,0.05,True
21,choice_only,log_beta,49,1,0.020408,0,1,0.05,False


In [ ]:
shared_boundary_triggered = bool(
    shared_boundary["trigger_fired"].any()
)

print("Any shared-parameter boundary trigger fired:",
      shared_boundary_triggered)

Any shared-parameter boundary trigger fired: True


In [ ]:
# 5.5 Inspect near-boundary fits

problematic_fits = boundary_detail[
    boundary_detail["near_boundary"]
].sort_values(
    ["parameter", "model", "participant"]
)

display(problematic_fits)

,model,participant,parameter,estimate,lower_bound,upper_bound,near_lower,near_upper,near_boundary
166,M1,4f8msviwg,log_b,-9.21034,-9.21034,4.60517,True,False,True
173,M1,52b64k0rp,log_b,-9.21034,-9.21034,4.60517,True,False,True
194,M1,8py7pm5i0,log_b,-9.21034,-9.21034,4.60517,True,False,True
208,M1,bh348lli7,log_b,-9.21034,-9.21034,4.60517,True,False,True
250,M1,dm8t9csqy,log_b,-9.21034,-9.21034,4.60517,True,False,True
...,...,...,...,...,...,...,...,...,...
87,choice_only,o7piq7dwr,log_k_R,4.60517,-4.60517,4.60517,False,True,True
93,choice_only,omy06qkfz,log_k_R,4.60517,-4.60517,4.60517,False,True,True
96,choice_only,orxx7nl6z,log_k_R,4.60517,-4.60517,4.60517,False,True,True
111,choice_only,rv0b4sx2y,log_k_R,-4.60517,-4.60517,4.60517,True,False,True


In [ ]:
# 5.6 Resolve fixed scale-aware log_b MAP prior centers

import numpy as np
import pandas as pd

from pd_project.valuation_choice import subjective_values
from pd_project.rt_models import rt_predictor


run_a_trials = trials.loc[
    trials["run"].astype(str) == "A"
].copy()

if run_a_trials.empty:
    raise RuntimeError("No Run-A trials available.")

s0 = float(
    config["data"]["transforms"]["amount_scale"]
)

# Fixed reference valuation: k_R = k_L = 1
reference_k = np.ones(
    len(run_a_trials),
    dtype=float,
)

values_ref = subjective_values(
    run_a_trials["r_cert"].to_numpy(dtype=float),
    run_a_trials["r_uncert"].to_numpy(dtype=float),
    run_a_trials["odds"].to_numpy(dtype=float),
    reference_k,
    s0=s0,
)

rows = []

for model in ("M1", "M2", "M3"):
    predictor = rt_predictor(
        model,
        values_ref.v_cert,
        values_ref.v_uncert,
        values_ref.delta_v,
    )

    predictor = np.asarray(predictor, dtype=float)
    predictor = predictor[np.isfinite(predictor)]

    if predictor.size == 0:
        raise RuntimeError(
            f"{model}: no finite RT predictor values."
        )

    g95 = float(np.quantile(predictor, 0.95))

    if not np.isfinite(g95) or g95 <= 0.0:
        raise RuntimeError(
            f"{model}: invalid predictor g95={g95}."
        )

    log_b_prior_mean = -float(np.log(g95))

    rows.append(
        {
            "model": model,
            "g95": g95,
            "log_b_prior_mean": log_b_prior_mean,
            "b_prior_median": float(
                np.exp(log_b_prior_mean)
            ),
        }
    )

log_b_prior_centers = pd.DataFrame(rows)

display(log_b_prior_centers)

## 6. Parameter Recovery

In [18]:
if RUN_FORMAL_RECOVERY:
    run_command([
        sys.executable,
        "scripts/run_recovery.py",
        "--config",
        "config/analysis.yaml",
        "--formal",
        "--confirm-design-frozen",
    ])
else:
    print("Formal recovery skipped. Set RUN_FORMAL_RECOVERY=True only after the recovery design is frozen.")

Formal recovery skipped. Set RUN_FORMAL_RECOVERY=True only after the recovery design is frozen.


In [19]:
recovery_dir = PROJECT_ROOT / config["outputs"]["recovery_directory"]
recovery_metrics_path = recovery_dir / "parameter_recovery_metrics.csv"
recovery_fits_path = recovery_dir / "fit_results.csv"
recovery_receipt_path = recovery_dir / "receipt.json"

if recovery_metrics_path.exists():
    recovery_metrics = pd.read_csv(recovery_metrics_path)
    display(recovery_metrics)
else:
    recovery_metrics = None
    print("Formal recovery metrics are not available yet.")

Formal recovery metrics are not available yet.


In [20]:
# Full-vs-choice-only recovery comparison for shared parameters
if recovery_metrics is not None:
    shared = recovery_metrics[
        recovery_metrics["parameter"].isin(["log_k_R", "log_k_L", "log_beta"])
    ].copy()

    full = shared[shared["fit_type"] == "generating_full"].copy()
    baseline = shared[shared["fit_type"] == "choice_only"].copy()

    paired = full.merge(
        baseline,
        on=["generating_model", "parameter"],
        suffixes=("_full", "_choice"),
    )
    paired["delta_rmse_full_minus_choice"] = (
        paired["rmse_full"] - paired["rmse_choice"]
    )
    paired["delta_nrmse_full_minus_choice"] = (
        paired["nrmse_full"] - paired["nrmse_choice"]
    )
    display(
        paired[
            [
                "generating_model",
                "parameter",
                "rmse_full",
                "rmse_choice",
                "delta_rmse_full_minus_choice",
                "nrmse_full",
                "nrmse_choice",
                "delta_nrmse_full_minus_choice",
            ]
        ]
    )

## 7. Model Recovery

In [21]:
confusion_path = recovery_dir / "model_recovery_confusion.csv"

if confusion_path.exists():
    model_recovery = pd.read_csv(confusion_path)
    display(model_recovery)
else:
    model_recovery = None
    print("Model-recovery confusion matrix is not available yet.")

Model-recovery confusion matrix is not available yet.


## 8. Pre-Run-B Freeze Summary

In [3]:
readiness = pd.DataFrame(
    [
        {"item": key, "ready": value}
        for key, value in config["project"]["pipeline_readiness"].items()
    ]
)

display(readiness)

print("freeze_status =", config["project"]["freeze_status"])
print("formal_run_b_enabled =", config["project"]["formal_run_b_enabled"])

gate2_ready = (
    config["project"]["freeze_status"] == "frozen"
    and config["project"]["formal_run_b_enabled"] is True
    and readiness["ready"].all()
)

print("Gate-2 machine-readable readiness:", bool(gate2_ready))

,item,ready
0,formal_parameter_recovery_implemented,True
1,formal_model_recovery_implemented,True
2,report_statistics_implemented,True
3,run_b_one_shot_smoke_tested,True
4,notebook_end_to_end_implemented,True
5,formal_artifact_archival_approved,True
6,remote_run_reservation_approved,True


freeze_status = frozen
formal_run_b_enabled = True
Gate-2 machine-readable readiness: True


In [4]:
status_path = PROJECT_ROOT / config["run_b_guard"]["status_registry"]
if status_path.exists():
    formal_status = json.loads(status_path.read_text(encoding="utf-8"))
else:
    formal_status = {"status": "missing"}

formal_status

{'status': 'not_run',
 'fingerprint': None,
 'note': 'This tracked registry is changed to in_progress before the one-shot run and completed only after success.'}

## 9. Formal Run-B Evaluation

**Do not casually execute this section.**

The formal command performs both:
1. held-out Run-B scoring from the frozen Run-A estimates, and
2. independently initialized Run-B fits used for test-retest reliability.

The underlying script enforces frozen configuration, fingerprints, a local lock,
and the tracked one-shot status registry.

In [13]:
# Safe guarded one-shot trigger.
# If formal results already exist, this cell reads them instead of rerunning.

status = formal_status.get("status")

if status == "completed":
    print("Formal Run-B already completed. Reusing frozen artifacts.")
elif RUN_FORMAL_RUN_B:
    if not gate2_ready:
        raise RuntimeError("Gate 2 is not fully frozen; refusing formal Run-B execution.")
    if status != "not_run":
        raise RuntimeError(f"Formal registry status is {status!r}, not 'not_run'.")
    run_command([
        sys.executable,
        "scripts/run_b_once.py",
        "--config",
        "config/analysis.yaml",
    ])
else:
    print(
        "Formal Run-B not executed. "
        "Set RUN_FORMAL_RUN_B=True only after Gate 2 approval."
    )

$ /opt/anaconda3/envs/ccs-pd/bin/python scripts/run_b_once.py --config config/analysis.yaml
Formal run-B outputs written once under /Users/zhengbinheng/Desktop/probability-discounting-choice-rt/results/formal_run_b


In [14]:
formal_dir = PROJECT_ROOT / "results" / "formal_run_b"
trial_scores_path = formal_dir / "trial_scores.csv"

if trial_scores_path.exists():
    trial_scores = pd.read_csv(trial_scores_path, dtype={"participant": "string"})
    print("Formal trial-score rows:", len(trial_scores))
    display(trial_scores.head())
else:
    trial_scores = None
    print("Formal Run-B scores are not available yet.")

Formal trial-score rows: 39200


,model,participant,condition,trial_index,choice_log_score,brier_score,choice_correct,rt_log_score,absolute_log_rt_error,absolute_rt_error_seconds,out_of_support
0,M1,2rrjzhgj9,L,1,-0.466979,0.139209,1.0,-2.112280,0.472500,1.415672,False
1,M1,2rrjzhgj9,L,2,-0.539878,0.174040,1.0,-0.689419,0.027937,0.064722,False
2,M1,2rrjzhgj9,L,3,-0.538214,0.173231,1.0,-0.692283,0.024217,0.056205,False
3,M1,2rrjzhgj9,L,4,-0.538543,0.173391,1.0,-0.731842,0.016621,0.039372,False
4,M1,2rrjzhgj9,L,5,-0.591775,0.199501,1.0,-0.766675,0.283867,0.581440,False


In [15]:
# Participant-first held-out summaries are generated by make_outputs.py.
if RUN_DERIVED_OUTPUTS:
    run_command([
        sys.executable,
        "scripts/make_outputs.py",
        "--config",
        "config/analysis.yaml",
    ])
else:
    print("Derived output generation skipped.")

Derived output generation skipped.


In [16]:
participant_scores_path = formal_dir / "participant_condition_scores.csv"
group_summary_path = formal_dir / "group_score_summary.csv"
comparisons_path = formal_dir / "model_comparisons.csv"

participant_scores = (
    pd.read_csv(participant_scores_path, dtype={"participant": "string"})
    if participant_scores_path.exists()
    else None
)
group_summary = (
    pd.read_csv(group_summary_path)
    if group_summary_path.exists()
    else None
)
model_comparisons = (
    pd.read_csv(comparisons_path)
    if comparisons_path.exists()
    else None
)

if group_summary is not None:
    display(group_summary)

## 10. Reliability

In [28]:
run_b_reliability_path = formal_dir / "run_b_reliability_fits.csv"

if run_a_fits is not None and run_b_reliability_path.exists():
    run_b_fits = pd.read_csv(
        run_b_reliability_path, dtype={"participant": "string"}
    )
    print("Run-B reliability fit rows:", len(run_b_fits))
else:
    run_b_fits = None
    print("A/B reliability inputs are not both available.")

A/B reliability inputs are not both available.


In [29]:
from scipy.stats import spearmanr
from pd_project.reliability import icc_a1, bland_altman_summary

reliability_rows = []

if run_a_fits is not None and run_b_fits is not None:
    for model in ("choice_only", "M1", "M2", "M3"):
        a = run_a_fits[
            (run_a_fits["model"] == model) & run_a_fits["success"].astype(bool)
        ]
        b = run_b_fits[
            (run_b_fits["model"] == model) & run_b_fits["success"].astype(bool)
        ]

        merged = a.merge(
            b,
            on="participant",
            suffixes=("_A", "_B"),
        )

        params = ["log_k_R", "log_k_L", "log_beta"]
        for parameter in params:
            x = merged[f"{parameter}_A"].to_numpy(float)
            y = merged[f"{parameter}_B"].to_numpy(float)
            valid = np.isfinite(x) & np.isfinite(y)

            if valid.sum() < 3:
                continue

            matrix = np.column_stack([x[valid], y[valid]])
            icc = icc_a1(matrix)
            rho = spearmanr(x[valid], y[valid]).statistic
            ba = bland_altman_summary(x[valid], y[valid])

            reliability_rows.append({
                "model": model,
                "parameter": parameter,
                "n": int(valid.sum()),
                "icc_a1": icc,
                "spearman": rho,
                **ba,
            })

reliability_summary = pd.DataFrame(reliability_rows)
reliability_summary

""


In [30]:
# Paired ΔICC: full model minus choice-only.
from pd_project.fit import deterministic_seed
from pd_project.reliability import paired_bootstrap_icc_difference

icc_difference_rows = []

if run_a_fits is not None and run_b_fits is not None:
    n_boot = int(config["bootstrap"]["resamples"])
    master_seed = int(config["random"]["master_seed"])

    for model in ("M1", "M2", "M3"):
        for parameter in ("log_k_R", "log_k_L", "log_beta"):
            a_full = run_a_fits[run_a_fits["model"] == model][
                ["participant", parameter]
            ].rename(columns={parameter: "full_A"})
            b_full = run_b_fits[run_b_fits["model"] == model][
                ["participant", parameter]
            ].rename(columns={parameter: "full_B"})
            a_choice = run_a_fits[run_a_fits["model"] == "choice_only"][
                ["participant", parameter]
            ].rename(columns={parameter: "choice_A"})
            b_choice = run_b_fits[run_b_fits["model"] == "choice_only"][
                ["participant", parameter]
            ].rename(columns={parameter: "choice_B"})

            joined = (
                a_full.merge(b_full, on="participant")
                .merge(a_choice, on="participant")
                .merge(b_choice, on="participant")
            )

            seed = deterministic_seed(
                master_seed, "icc_difference", model, parameter
            )

            result = paired_bootstrap_icc_difference(
                joined,
                participant_column="participant",
                full_a_column="full_A",
                full_b_column="full_B",
                choice_a_column="choice_A",
                choice_b_column="choice_B",
                n_boot=n_boot,
                seed=seed,
            )

            icc_difference_rows.append(
                {"model": model, "parameter": parameter, **result}
            )

delta_icc = pd.DataFrame(icc_difference_rows)
delta_icc

""


## 11. Does RT Help?

In [31]:
# Held-out choice improvement is the primary predictive answer.
# Positive values mean full RT-informed fitting predicts Run-B choices better.

if model_comparisons is not None:
    rt_help_choice = model_comparisons[
        model_comparisons["family"] == "rt_informed_vs_choice_only"
    ].copy()
    display(rt_help_choice)
else:
    rt_help_choice = None
    print("Run make_outputs.py after formal Run-B to obtain choice contrasts.")

Run make_outputs.py after formal Run-B to obtain choice contrasts.


In [32]:
# Recovery improvement: negative ΔRMSE / ΔNRMSE means RT-informed fitting recovered
# the shared parameter more accurately than choice-only fitting.
if recovery_metrics is not None:
    display(
        paired[
            [
                "generating_model",
                "parameter",
                "delta_rmse_full_minus_choice",
                "delta_nrmse_full_minus_choice",
            ]
        ]
    )

# Reliability improvement
if 'delta_icc' in globals() and not delta_icc.empty:
    display(delta_icc)

## 12. Support-Shift Diagnostics

In [33]:
if trial_scores is not None and "out_of_support" in trial_scores.columns:
    full_scores = trial_scores[trial_scores["model"].isin(["M1", "M2", "M3"])].copy()

    support_summary = (
        full_scores.groupby(["model", "condition"], as_index=False)
        .agg(
            out_of_support_trial_fraction=("out_of_support", "mean"),
            n_trials=("out_of_support", "size"),
        )
    )
    display(support_summary)

    participant_support = (
        full_scores.groupby(["model", "participant"])["out_of_support"]
        .any()
        .groupby("model")
        .mean()
        .rename("affected_participant_fraction")
        .reset_index()
    )
    display(participant_support)

    if "rt_log_score" in full_scores.columns:
        support_scores = (
            full_scores.dropna(subset=["rt_log_score"])
            .groupby(["model", "condition", "out_of_support"])["rt_log_score"]
            .mean()
            .rename("mean_rt_log_score")
            .reset_index()
        )
        display(support_scores)
else:
    print("Support-shift flags require formal Run-B trial scores.")

Support-shift flags require formal Run-B trial scores.


## 13. Bootstrap & Multiple Comparisons

In [34]:
# make_outputs.py implements participant-level paired bootstrap and Holm correction
# for the two frozen comparison families.

if model_comparisons is not None:
    display(model_comparisons)
else:
    print("No formal comparison table available yet.")

No formal comparison table available yet.


## 14. Final Figures & Tables

In [35]:
# This section intentionally reads frozen artifacts rather than recomputing model fits.
# Add publication-quality figures here after the numerical pipeline is complete.

final_table_paths = {
    "recovery": recovery_metrics_path,
    "model_recovery": confusion_path,
    "group_scores": group_summary_path,
    "comparisons": comparisons_path,
}

pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
        }
        for name, path in final_table_paths.items()
    ]
)

,artifact,path,exists
0,recovery,results/recovery/parameter_recovery_metrics.csv,False
1,model_recovery,results/recovery/model_recovery_confusion.csv,False
2,group_scores,results/formal_run_b/group_score_summary.csv,False
3,comparisons,results/formal_run_b/model_comparisons.csv,False


### Recommended final figure set

1. **Parameter recovery** — true vs estimated or recovery-error summary for shared parameters.
2. **Model recovery** — M1/M2/M3 confusion matrix.
3. **Reliability** — A vs B parameter estimates and/or ICC summary.
4. **Run-B predictive performance** — choice and RT MLPD by condition.
5. **Value of RT modality** — full minus choice-only choice MLPD and recovery/reliability changes.
6. **Support shift** — in-support vs out-of-support predictive scores.

Keep report figures composite/multi-panel because the course requires figures after the five-page main body.

## 15. Reproducibility / Artifact Manifest

In [36]:
manifest_path = PROJECT_ROOT / config["outputs"]["manifest"]

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    display(manifest.tail(30))
else:
    manifest = None
    print("Manifest does not exist yet.")

,artifact,stage,timestamp_utc,git_commit,config_sha256,raw_data_sha256,processed_data_sha256,data_pipeline_sha256,raw_archive_sha256,raw_source_mode,artifact_sha256,seed,participant,run,condition,model,fit_status,path
0,processed_trials,prepare_data,2026-08-17T21:00:58.937952+00:00,NaN,9f49ea69bea6665ba15c4370c4225d67c25a1fc52202fe...,7f6b760de3f216e6eefd8024c82a5629fad721aadedf5a...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,d17493a45aec0b158be76cafbee9cac4fdf099a3492c11...,NaN,direct_mat,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,NaN,NaN,NaN,NaN,NaN,NaN,data/processed/pd_trials.csv
1,data_audit,prepare_data,2026-08-17T21:00:58.937952+00:00,NaN,9f49ea69bea6665ba15c4370c4225d67c25a1fc52202fe...,7f6b760de3f216e6eefd8024c82a5629fad721aadedf5a...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,d17493a45aec0b158be76cafbee9cac4fdf099a3492c11...,NaN,direct_mat,cd0e27d57ef5f9ab76cc39dd9a5e0e5efb9b96630c40fa...,NaN,NaN,NaN,NaN,NaN,passed,data/processed/data_audit.json
2,processed_trials,prepare_data,2026-08-17T21:47:29.856588+00:00,NaN,9f49ea69bea6665ba15c4370c4225d67c25a1fc52202fe...,7f6b760de3f216e6eefd8024c82a5629fad721aadedf5a...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,760dbdcb3f37cc780caf5cac0f2816070d099beca3c9c6...,NaN,direct_mat,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,NaN,NaN,NaN,NaN,NaN,NaN,data/processed/pd_trials.csv
3,data_audit,prepare_data,2026-08-17T21:47:29.856588+00:00,NaN,9f49ea69bea6665ba15c4370c4225d67c25a1fc52202fe...,7f6b760de3f216e6eefd8024c82a5629fad721aadedf5a...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,760dbdcb3f37cc780caf5cac0f2816070d099beca3c9c6...,NaN,direct_mat,c7bbe14cbab9fda887610000822088d344a7037275cf80...,NaN,NaN,NaN,NaN,NaN,passed,data/processed/data_audit.json
4,processed_trials,prepare_data,2026-08-17T22:12:44.219847+00:00,NaN,2b171995d7ba6736f44f922371a8192b0631c5e513bcdf...,7f6b760de3f216e6eefd8024c82a5629fad721aadedf5a...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,ad8b23c6b44ff33f2658dca31d7f9a47e68d03d6357923...,NaN,direct_mat,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,NaN,NaN,NaN,NaN,NaN,NaN,data/processed/pd_trials.csv
5,data_audit,prepare_data,2026-08-17T22:12:44.219847+00:00,NaN,2b171995d7ba6736f44f922371a8192b0631c5e513bcdf...,7f6b760de3f216e6eefd8024c82a5629fad721aadedf5a...,e2bb506ef9bd17c4b8e062646715087a07621486b7af13...,ad8b23c6b44ff33f2658dca31d7f9a47e68d03d6357923...,NaN,direct_mat,1bc0b7872aa8813f8a92c9f3abede6457ec8b10b0d50be...,NaN,NaN,NaN,NaN,NaN,passed,data/processed/data_audit.json


In [37]:
# Final artifact status snapshot
artifact_status = {
    "processed_data": processed_path.exists(),
    "data_audit": audit_path.exists(),
    "run_a_fits": run_a_fits_path.exists(),
    "formal_recovery": recovery_metrics_path.exists(),
    "model_recovery": confusion_path.exists(),
    "formal_run_b_scores": trial_scores_path.exists(),
    "run_b_reliability": run_b_reliability_path.exists(),
    "participant_condition_scores": participant_scores_path.exists(),
    "model_comparisons": comparisons_path.exists(),
    "manifest": manifest_path.exists(),
}

pd.Series(artifact_status, name="ready")

processed_data                   True
data_audit                       True
run_a_fits                      False
formal_recovery                 False
model_recovery                  False
formal_run_b_scores             False
run_b_reliability               False
participant_condition_scores    False
model_comparisons               False
manifest                         True
Name: ready, dtype: bool

## Final interpretation checkpoint

Do not write the report's substantive conclusion until:

- formal recovery has been reviewed;
- model recovery has been reviewed;
- Gate 2 is frozen;
- formal Run-B status is `completed`;
- participant-level predictive contrasts are generated;
- reliability diagnostics are complete;
- all reported numbers agree with the frozen output artifacts.

The notebook should finish as a reproducible presentation layer over the same tested code used by the command-line pipeline.